# 🧪 Engineering AI Coding Agents
## From Code Autocomplete to Autonomous Multi-Agent Software Teams

Welcome to the comprehensive laboratory for engineering modern AI software development systems.

In this notebook, we simulate the engineering organization of **NovaStack**, a fast-growing developer tools startup building **DevFlow**—an AI-powered developer productivity platform.

```text
Code Completion
      ↓
AI Coding Assistant (Prompt ➔ Code)
      ↓
Repository-Aware Coding Agent
      ↓
Terminal / CLI Coding Agent
      ↓
Self-Correcting Coding Agent
      ↓
AI Code Review Agent
      ↓
Multi-Agent Engineering Team
      ↓
Agentic CI/CD & Human Governance
```

### Teaching Philosophy
Every major concept follows:
> **Problem → Naive Approach → Limitation → Agentic Approach → Implementation → Observation → Engineering Takeaway**

---
# 00 — Environment & Architecture

### What We Are Building
We are building a local, fully functional **AI Software Engineering Team** that interacts with an actual software repository on disk.

Unlike simple code autocompletion (which merely suggests the next few tokens based on open editor tabs), an **Autonomous Coding Agent**:
1. Inspects the directory structure and reads project documentation (`AGENTS.md`, architecture guidelines).
2. Decomposes tasks into structured plans.
3. Edits files, executes terminal commands, and runs `pytest` test suites.
4. Detects test failures, inspects tracebacks, and self-corrects its code.
5. Collaborates across specialized roles (Supervisor, Developer, Reviewer, QA).
6. Requests human sign-off before executing irreversible actions (such as production deployments).

### System Architecture
```text
                         HUMAN ENGINEER
                               │
                               ▼
                        ┌─────────────┐
                        │ SUPERVISOR  │
                        │    AGENT    │
                        └──────┬──────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
       ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
       │  RESEARCH   │  │  DEVELOPER  │  │     QA      │
       │    AGENT    │  │    AGENT    │  │    AGENT    │
       └──────┬──────┘  └──────┬──────┘  └──────┬──────┘
              │                │                │
              └────────────────┼────────────────┘
                               ▼
                        ┌─────────────┐
                        │  REVIEWER   │
                        │    AGENT    │
                        └──────┬──────┘
                               │
                               ▼
                        ┌─────────────┐
                        │ TOOL LAYER  │
                        └──────┬──────┘
                               │
       ┌───────────┬───────────┼───────────┬───────────┐
       ▼           ▼           ▼           ▼           ▼
   ┌───────┐  ┌──────────┐ ┌───────┐  ┌─────────┐ ┌─────────┐
   │ Files │  │ Terminal │ │  Git  │  │ Pytest  │ │  CI/CD  │
   └───┬───┘  └────┬─────┘ └───┬───┘  └────┬────┘ └────┬────┘
       │           │           │           │           │
       └───────────┴───────────┼───────────┴───────────┘
                               ▼
                     ┌───────────────────┐
                     │   OBSERVABILITY   │
                     │  (Events/Traces)  │
                     └─────────┬─────────┘
                               │
                               ▼
                     ┌───────────────────┐
                     │   POLICY ENGINE   │
                     │  (HITL Clearance) │
                     └─────────┬─────────┘
                               │
                               ▼
                     ┌───────────────────┐
                     │  HUMAN APPROVAL   │
                     │ (Production Gate) │
                     └─────────┬─────────┘
                               │
                               ▼
                           DEPLOYMENT
```

In [1]:
# 00.1 System Imports and Initialization
import os
import sys
import time
import json
import re
import uuid
import shutil
import subprocess
import tempfile
import logging
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Any, Optional, Callable, Union, Tuple
from datetime import datetime
from pathlib import Path

# Scientific and structured data packages
import pandas as pd
import numpy as np
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("DevFlowAgentEngine")

print(f"✅ Python Runtime: {sys.version.split()[0]}")
print("✅ Core Frameworks Loaded: Pydantic v2, Pytest, Pandas, Subprocess")

✅ Python Runtime: 3.13.7
✅ Core Frameworks Loaded: Pydantic v2, Pytest, Pandas, Subprocess


In [2]:
# 00.2 Unified LLM Client Architecture (Live Frontier API or Zero-Cost Simulator)
@dataclass
class CodingLLMConfig:
    provider: str = field(default_factory=lambda: os.getenv("LLM_PROVIDER", "mock"))
    model_name: str = field(default_factory=lambda: os.getenv("OPENAI_MODEL", "gpt-4o"))
    temperature: float = 0.1
    max_tokens: int = 1500

class CodingLLMResponse(BaseModel):
    content: str
    tool_calls: List[Dict[str, Any]] = []
    input_tokens: int = 0
    output_tokens: int = 0
    latency_ms: float = 0.0

class DeterministicCodingMockEngine:
    """
    High-fidelity deterministic coding engine that simulates realistic model responses,
    tool call generation, and architectural plans without requiring paid external APIs.
    """
    def __init__(self):
        self.call_count = 0

    def generate(self, messages: List[Dict[str, str]], tools: Optional[List[Dict[str, Any]]] = None) -> CodingLLMResponse:
        self.call_count += 1
        last_msg = messages[-1]["content"].lower() if messages else ""
        start_time = time.time()
        tool_calls = []
        content = ""

        # Semantic tool dispatch simulation
        if tools:
            if "list" in last_msg or "structure" in last_msg:
                tool_calls.append({
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "function": {
                        "name": "list_files",
                        "arguments": json.dumps({"directory": "."})
                    }
                })
                content = "I will inspect the directory structure of the repository."
            elif "search" in last_msg or "where" in last_msg or "find" in last_msg:
                tool_calls.append({
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "function": {
                        "name": "search_files",
                        "arguments": json.dumps({"query": "notification", "file_pattern": "*.py"})
                    }
                })
                content = "I am searching for notification implementations across python files."
            elif "read" in last_msg or "inspect" in last_msg:
                target_file = "AGENTS.md" if "agents" in last_msg else "backend/services/notification.py"
                tool_calls.append({
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "function": {
                        "name": "read_file",
                        "arguments": json.dumps({"path": target_file})
                    }
                })
                content = f"Reading {target_file} to understand guidelines and code."
            elif "test" in last_msg or "pytest" in last_msg:
                tool_calls.append({
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "function": {
                        "name": "run_tests",
                        "arguments": json.dumps({"test_path": "tests/test_notification.py"})
                    }
                })
                content = "Running pytest to verify service functionality."

        if not tool_calls:
            if "create a python function for sending notifications" in last_msg:
                content = (
                    "def send_notification(user_id: str, message: str) -> bool:\n"
                    "    # Naive assistant generation without repo context\n"
                    "    print(f'Sending {message} to {user_id}')\n"
                    "    return True"
                )
            elif "explain how they work" in last_msg:
                content = (
                    "### Notification System Architecture Summary:\n"
                    "- Implemented in `backend/services/notification.py` via `NotificationService`.\n"
                    "- Emits messages to in-app queues and dispatches email via SMTP client.\n"
                    "- Enforces tenant isolation and rates per NovaStack `AGENTS.md`."
                )
            else:
                content = (
                    "Task execution completed in accordance with NovaStack architectural standards. "
                    "All test assertions passed and git diff is verified."
                )

        latency = round((time.time() - start_time + 0.045) * 1000, 2)
        in_tokens = len(str(messages)) // 4 + 45
        out_tokens = len(content) // 4 + (len(str(tool_calls)) // 4) + 20

        return CodingLLMResponse(
            content=content,
            tool_calls=tool_calls,
            input_tokens=in_tokens,
            output_tokens=out_tokens,
            latency_ms=latency
        )

global_coding_mock = DeterministicCodingMockEngine()

def run_coding_llm(messages: List[Dict[str, str]], tools: Optional[List[Dict[str, Any]]] = None) -> CodingLLMResponse:
    """Unified interface for coding LLM invocations."""
    return global_coding_mock.generate(messages=messages, tools=tools)

print("🚀 Coding LLM Engine Initialized (Mode='mock': 100% deterministic local execution)")

🚀 Coding LLM Engine Initialized (Mode='mock': 100% deterministic local execution)


---
# 01 — Create a Realistic Software Repository

To teach repository-aware agent engineering, we programmatically construct an actual software repository on disk under a sandboxed directory:

```text
devflow/
├── frontend/
│   ├── components/
│   └── services/
├── backend/
│   ├── api/
│   ├── services/
│   └── models/
├── tests/
├── scripts/
├── docs/
│   ├── architecture.md
│   ├── coding_standards.md
│   └── decisions.md
├── config/
├── README.md
├── AGENTS.md
├── tasks.md
└── pyproject.toml
```

We include **intentional bugs** (such as an unhandled payload exception in `backend/services/notification.py`) that our agents will discover and fix.

In [3]:
# 01.1 Programmatic Sandbox Repository Creation
SANDBOX_DIR = Path(tempfile.mkdtemp(prefix="devflow_repo_")).resolve()
print(f"📁 Creating DevFlow Sandbox at: {SANDBOX_DIR}")

# Directory Structure Definition
directories = [
    SANDBOX_DIR / "frontend" / "components",
    SANDBOX_DIR / "frontend" / "services",
    SANDBOX_DIR / "backend" / "api",
    SANDBOX_DIR / "backend" / "services",
    SANDBOX_DIR / "backend" / "models",
    SANDBOX_DIR / "tests",
    SANDBOX_DIR / "scripts",
    SANDBOX_DIR / "docs",
    SANDBOX_DIR / "config"
]

for d in directories:
    d.mkdir(parents=True, exist_ok=True)

# 1. Project Configuration & Memory
(SANDBOX_DIR / "pyproject.toml").write_text("""[project]
name = "devflow"
version = "0.1.0"
description = "DevFlow: AI-Powered Developer Productivity Platform"
dependencies = ["pydantic>=2.0.0", "pytest>=8.0.0"]
""")

(SANDBOX_DIR / "AGENTS.md").write_text("""# DevFlow Agent Operating Rules
1. Always add type hints to all Python functions.
2. Business logic must reside in `backend/services/`, never in `backend/api/`.
3. Never swallow exceptions silently; log them with correlation IDs.
4. Run `pytest` and verify 100% pass rate before requesting code review.
5. Production deployment requires explicit human sign-off.
""")

(SANDBOX_DIR / "docs" / "architecture.md").write_text("""# DevFlow System Architecture
- Clean Architecture / Ports and Adapters.
- NotificationService handles multi-channel dispatch (Email, In-App).
- PostgreSQL with pgvector is primary persistence.
""")

(SANDBOX_DIR / "tasks.md").write_text("""# Engineering Sprint Tasks
- [x] Scaffold repository architecture
- [ ] Fix NotificationService payload validation bug
- [ ] Add Email and In-App multi-channel delivery support
- [ ] Run test suite and pass all assertions
""")

# 2. Application Source Code (With an intentional bug!)
(SANDBOX_DIR / "backend" / "models" / "user.py").write_text("""from pydantic import BaseModel, EmailStr

class User(BaseModel):
    id: str
    name: str
    email: str
    is_active: bool = True
""")

(SANDBOX_DIR / "backend" / "services" / "notification.py").write_text("""# Notification Service Implementation
from typing import Dict, Any

class NotificationService:
    def __init__(self):
        self.sent_history = []

    def dispatch(self, user_id: str, payload: Dict[str, Any]) -> bool:
        # BUG: Intentionally missing validation for empty message content!
        # When payload['message'] is missing or None, raises KeyError instead of ValueError
        message_body = payload["message"]
        
        record = {
            "user_id": user_id,
            "message": message_body,
            "channel": payload.get("channel", "in_app"),
            "status": "DELIVERED"
        }
        self.sent_history.append(record)
        return True
""")

# 3. Test Suite
(SANDBOX_DIR / "tests" / "test_notification.py").write_text("""import pytest
from backend.services.notification import NotificationService

def test_successful_notification():
    service = NotificationService()
    res = service.dispatch("usr_123", {"message": "PR #42 approved!", "channel": "in_app"})
    assert res is True
    assert len(service.sent_history) == 1

def test_missing_message_raises_value_error():
    service = NotificationService()
    # Expects ValueError on empty or missing payload, but the buggy implementation raises KeyError!
    with pytest.raises(ValueError):
        service.dispatch("usr_123", {})
""")

print(f"✅ Sandbox Repository Successfully Created with {len(list(SANDBOX_DIR.rglob('*')))} files & folders.")

📁 Creating DevFlow Sandbox at: /private/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/devflow_repo_c6s66v77
✅ Sandbox Repository Successfully Created with 18 files & folders.


### 💡 Engineering Takeaway — Part 01
Autonomous coding agents must operate inside isolated, reproducible sandbox environments. Never allow an agent to experiment directly on your root development machine or uncommitted production repositories.

---
# 02 — AI Coding Assistant (Prompt ➔ Code)

### The Naive Approach
The standard pattern used by basic chat models is a single prompt-response pass:
```text
User ──► Prompt ──► LLM ──► Generated Code Snippet
```

In [4]:
# 02.1 Bare Coding Assistant Prompt Call
naive_prompt = "Create a Python function for sending notifications"

res = run_coding_llm(messages=[
    {"role": "user", "content": naive_prompt}
])

print("=== NAIVE CODING ASSISTANT RESPONSE ===")
print(res.content)

=== NAIVE CODING ASSISTANT RESPONSE ===
def send_notification(user_id: str, message: str) -> bool:
    # Naive assistant generation without repo context
    print(f'Sending {message} to {user_id}')
    return True


### Why Is This Insufficient?
Notice what this code snippet is missing:
1. **No Repository Awareness**: It created a detached function from scratch, unaware that `backend/services/notification.py` and `NotificationService` already exist.
2. **No Coding Standards**: It ignored NovaStack's `AGENTS.md` guidelines.
3. **No Execution or Verification**: It has no idea if the code actually runs, satisfies dependencies, or passes existing test suites.
4. **No Feedback Loop**: If it contains bugs, it has no mechanism to observe runtime exceptions or fix them.

### Progression: From Code Autocomplete to Autonomous Agent

In [5]:
# 02.2 Paradigm Comparison Matrix
evolution_table = [
    {"Stage": "1. Code Autocomplete", "Context": "Current line / token prefix", "Action": "Next token prediction", "Verification": "None"},
    {"Stage": "2. AI Chat Assistant", "Context": "User prompt + pasted snippet", "Action": "Generates markdown code block", "Verification": "User manually copies & runs"},
    {"Stage": "3. Repository Agent", "Context": "File tree + grep + AST", "Action": "Direct file read & write", "Verification": "Syntax validation"},
    {"Stage": "4. Autonomous Team", "Context": "Full repo, Git, docs, tests", "Action": "Plan ➔ Edit ➔ Test ➔ Review", "Verification": "Automated Pytest & CI/CD"}
]

df_evol = pd.DataFrame(evolution_table)
print(df_evol.to_string(index=False))

               Stage                      Context                        Action                Verification
1. Code Autocomplete  Current line / token prefix         Next token prediction                        None
2. AI Chat Assistant User prompt + pasted snippet Generates markdown code block User manually copies & runs
 3. Repository Agent       File tree + grep + AST      Direct file read & write           Syntax validation
  4. Autonomous Team  Full repo, Git, docs, tests   Plan ➔ Edit ➔ Test ➔ Review    Automated Pytest & CI/CD


### 💡 Engineering Takeaway — Part 02
*Code generation is only 20% of software engineering.* The remaining 80% consists of understanding legacy context, coordinating dependencies, testing edge cases, and verifying compliance with team standards.

---
# 03 — Repository-Aware Coding Agent

To make an agent repository-aware, we equip it with file-system inspection tools:
- `list_files(directory)`
- `read_file(path)`
- `search_files(query, file_pattern)`

Let us demonstrate the task:
> **“Find where user notifications are implemented in DevFlow and explain how they work.”**

In [6]:
# 03.1 File System Inspection Tools
def tool_list_files(directory: str = ".") -> List[str]:
    """Lists files in the sandbox repository relative to repo root."""
    target = (SANDBOX_DIR / directory).resolve()
    rel_files = []
    base_dir = SANDBOX_DIR.resolve()
    for p in target.rglob("*"):
        if p.is_file() and not p.name.startswith("."):
            rel_files.append(str(p.resolve().relative_to(base_dir)))
    return sorted(rel_files)

def tool_read_file(path: str) -> str:
    """Reads the complete content of a file within the sandbox repository."""
    target = (SANDBOX_DIR / path).resolve()
    if not target.exists():
        return f"Error: File '{path}' does not exist."
    return target.read_text(encoding="utf-8")

def tool_search_files(query: str, file_pattern: str = "*.py") -> List[Dict[str, Any]]:
    """Searches for a text pattern across repository files."""
    matches = []
    base_dir = SANDBOX_DIR.resolve()
    for p in SANDBOX_DIR.rglob(file_pattern):
        if p.is_file():
            text = p.read_text(encoding="utf-8", errors="ignore")
            for idx, line in enumerate(text.splitlines(), start=1):
                if query.lower() in line.lower():
                    matches.append({
                        "file": str(p.resolve().relative_to(base_dir)),
                        "line": idx,
                        "content": line.strip()
                    })
    return matches

print("🔎 Repository Files Discovered:")
for f in tool_list_files()[:8]:
    print(f"  • {f}")

print("\n🔎 Searching for 'notification':")
search_hits = tool_search_files("notification")
for hit in search_hits[:3]:
    print(f"  • {hit['file']}:{hit['line']} ➔ {hit['content']}")

🔎 Repository Files Discovered:
  • AGENTS.md
  • backend/models/user.py
  • backend/services/notification.py
  • docs/architecture.md
  • pyproject.toml
  • tasks.md
  • tests/test_notification.py

🔎 Searching for 'notification':
  • tests/test_notification.py:2 ➔ from backend.services.notification import NotificationService
  • tests/test_notification.py:4 ➔ def test_successful_notification():
  • tests/test_notification.py:5 ➔ service = NotificationService()


In [7]:
# 03.2 Repository-Aware Investigation Workflow
def run_repo_investigation(task: str) -> Dict[str, Any]:
    trace = []
    # 1. Search for notification references
    hits = tool_search_files("notification")
    trace.append({"action": "search_files", "query": "notification", "hits_found": len(hits)})
    
    # 2. Read identified file
    target_file = hits[0]["file"]
    content = tool_read_file(target_file)
    trace.append({"action": "read_file", "file": target_file, "size_chars": len(content)})
    
    # 3. Query LLM with grounded file content
    prompt = f"""
You are a senior engineer at NovaStack. Explain how notifications work based on this file:
=== FILE: {target_file} ===
{content}

Task: {task}
"""
    res = run_coding_llm(messages=[{"role": "user", "content": prompt}])
    
    return {
        "task": task,
        "trace": trace,
        "summary": res.content
    }

investigation = run_repo_investigation("Find where user notifications are implemented and explain how they work.")
print("📋 Investigation Result:")
print(investigation["summary"])

📋 Investigation Result:
### Notification System Architecture Summary:
- Implemented in `backend/services/notification.py` via `NotificationService`.
- Emits messages to in-app queues and dispatches email via SMTP client.
- Enforces tenant isolation and rates per NovaStack `AGENTS.md`.


### 💡 Engineering Takeaway — Part 03
Grounding models in repository context eliminates hallucinated imports and prevents the agent from reinventing existing services.

---
# 04 — Structured Tool Calling with Pydantic

Allowing models to emit raw, unstructured text strings leads to parse failures and security vulnerabilities.
We use **Pydantic v2** to define typed, validated schemas for all agent tool invocations:
- `ReadFileArgs`
- `WriteFileArgs`
- `SearchFilesArgs`
- `ListFilesArgs`
- `RunCommandArgs`
- `RunTestsArgs`

In [8]:
# 04.1 Typed Tool Schemas using Pydantic v2
class ReadFileArgs(BaseModel):
    path: str = Field(description="Relative path of file to read from repository root.")

class WriteFileArgs(BaseModel):
    path: str = Field(description="Relative path of file to write.")
    content: str = Field(description="Full text content to write into file.")

class SearchFilesArgs(BaseModel):
    query: str = Field(description="Text string to search for across repo.")
    file_pattern: Optional[str] = Field(default="*.py", description="Glob pattern filter.")

class ListFilesArgs(BaseModel):
    directory: Optional[str] = Field(default=".", description="Target directory path.")

class RunCommandArgs(BaseModel):
    command: str = Field(description="Shell command to execute within sandbox.")

class RunTestsArgs(BaseModel):
    test_path: Optional[str] = Field(default="tests", description="Path to test directory or file.")

print("✅ Pydantic Tool Schemas Defined: ReadFile, WriteFile, SearchFiles, ListFiles, RunCommand, RunTests")

✅ Pydantic Tool Schemas Defined: ReadFile, WriteFile, SearchFiles, ListFiles, RunCommand, RunTests


In [9]:
# 04.2 Tool Dispatch Registry with Strict Validation
def tool_write_file(args: WriteFileArgs) -> str:
    target = (SANDBOX_DIR / args.path).resolve()
    # Directory boundary check to prevent path traversal
    if not str(target).startswith(str(SANDBOX_DIR.resolve())):
        raise PermissionError(f"Security Alert: Attempted path traversal to '{args.path}'.")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(args.content, encoding="utf-8")
    return f"Successfully wrote {len(args.content)} characters to '{args.path}'."

TOOL_REGISTRY: Dict[str, Tuple[Callable, type]] = {
    "list_files": (lambda args: tool_list_files(args.directory), ListFilesArgs),
    "read_file": (lambda args: tool_read_file(args.path), ReadFileArgs),
    "search_files": (lambda args: tool_search_files(args.query, args.file_pattern), SearchFilesArgs),
    "write_file": (tool_write_file, WriteFileArgs)
}

def execute_structured_tool(name: str, raw_arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Validates arguments against Pydantic schema before executing tool."""
    if name not in TOOL_REGISTRY:
        return {"status": "ERROR", "message": f"Tool '{name}' not found."}
    
    fn, schema_cls = TOOL_REGISTRY[name]
    try:
        validated = schema_cls(**raw_arguments)
        result = fn(validated)
        return {"status": "SUCCESS", "output": result}
    except ValidationError as ve:
        return {"status": "VALIDATION_ERROR", "error": ve.errors()}
    except Exception as e:
        return {"status": "EXECUTION_ERROR", "error": str(e)}

# Demonstrate validation defense against malformed model output
bad_payload = {"path": 12345}  # Missing content, invalid type
val_test = execute_structured_tool("write_file", bad_payload)
print("🛡️ Pydantic Validation Defense in Action:")
print(json.dumps(val_test, indent=2, default=str))

🛡️ Pydantic Validation Defense in Action:
{
  "status": "VALIDATION_ERROR",
  "error": [
    {
      "type": "string_type",
      "loc": [
        "path"
      ],
      "msg": "Input should be a valid string",
      "input": 12345,
      "url": "https://errors.pydantic.dev/2.13/v/string_type"
    },
    {
      "type": "missing",
      "loc": [
        "content"
      ],
      "msg": "Field required",
      "input": {
        "path": 12345
      },
      "url": "https://errors.pydantic.dev/2.13/v/missing"
    }
  ]
}


### 💡 Engineering Takeaway — Part 04
Typed schemas act as an active firewall between model hallucinations and file-system mutations.

---
# 05 — Terminal / CLI Coding Agent

Giving an agent terminal access transforms it into an environment-interacting system.

### Mandatory Terminal Guardrails:
1. **Directory Boundary**: The subprocess must run strictly inside `SANDBOX_DIR`.
2. **Command Allowlist**: Only permit safe developer commands (`python`, `pytest`, `git`, `ls`).
3. **Execution Timeout**: Prevent infinite loops or hanging processes with a 15-second cutoff.
4. **Output Capture**: Separate stdout, stderr, and exit codes.

In [10]:
# 05.1 Safe Sandboxed Subprocess Runner
COMMAND_ALLOWLIST = {"python", "python3", "pytest", "git", "ls", "find"}

@dataclass
class TerminalResult:
    command: str
    exit_code: int
    stdout: str
    stderr: str
    duration_ms: float
    timestamp: str

def safe_run_terminal(command_str: str, timeout_seconds: int = 15) -> TerminalResult:
    tokens = command_str.strip().split()
    if not tokens:
        return TerminalResult(command_str, -1, "", "Empty command string.", 0.0, datetime.utcnow().isoformat())
    
    binary = tokens[0]
    if binary not in COMMAND_ALLOWLIST:
        return TerminalResult(
            command=command_str,
            exit_code=126,
            stdout="",
            stderr=f"Security Violation: Binary '{binary}' is not permitted in allowlist {COMMAND_ALLOWLIST}.",
            duration_ms=0.0,
            timestamp=datetime.utcnow().isoformat()
        )
    
    start_time = time.time()
    try:
        env = os.environ.copy()
        env["PATH"] = f"{os.path.dirname(sys.executable)}:{env.get('PATH', '')}"
        env["PYTHONPATH"] = str(SANDBOX_DIR.resolve())
        proc = subprocess.run(
            command_str,
            shell=True,
            cwd=SANDBOX_DIR,
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            env=env
        )
        elapsed_ms = round((time.time() - start_time) * 1000, 2)
        return TerminalResult(
            command=command_str,
            exit_code=proc.returncode,
            stdout=proc.stdout.strip(),
            stderr=proc.stderr.strip(),
            duration_ms=elapsed_ms,
            timestamp=datetime.utcnow().isoformat()
        )
    except subprocess.TimeoutExpired:
        return TerminalResult(
            command=command_str,
            exit_code=124,
            stdout="",
            stderr=f"Timeout: Command exceeded {timeout_seconds}s limit.",
            duration_ms=round((time.time() - start_time) * 1000, 2),
            timestamp=datetime.utcnow().isoformat()
        )

# Test safe command execution
term_res = safe_run_terminal("pytest tests/test_notification.py")
print(f"💻 Terminal Command Executed: {term_res.command}")
print(f"Exit Code: {term_res.exit_code} (Duration: {term_res.duration_ms}ms)")
print("Stdout Snippet:\n", term_res.stdout[:280] + "...")

# Test security blocking of unauthorized commands
blocked_res = safe_run_terminal("rm -rf /tmp/something")
print(f"\n🚫 Blocked Command Test: {blocked_res.stderr}")

💻 Terminal Command Executed: pytest tests/test_notification.py
Exit Code: 1 (Duration: 284.94ms)
Stdout Snippet:
 ============================= test session starts ==============================
platform darwin -- Python 3.13.7, pytest-9.1.1, pluggy-1.6.0
rootdir: /private/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/devflow_repo_c6s66v77
configfile: pyproject.toml
collected 2 ite...

🚫 Blocked Command Test: Security Violation: Binary 'rm' is not permitted in allowlist {'python', 'pytest', 'git', 'python3', 'find', 'ls'}.


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()
/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 05
Never pass unvetted strings directly to `/bin/sh`. Always enforce an allowlist of trusted binaries and execute within sandboxed working directories with strict execution timeouts.

---
# 06 — The Fundamental Agent Loop

The core architecture of every autonomous coding agent is the **Reason-Act-Observe-Verify** feedback loop:

```text
       ┌───────────────────────────────┐
       ▼                               │
┌─────────────┐                 ┌──────────────┐
│    PLAN     │                 │   OBSERVE    │
│ (Reasoning) │                 │(Test Output) │
└──────┬──────┘                 └──────▲───────┘
       │                               │
       ▼ Tool / Command                │
┌─────────────┐                        │
│     ACT     │────────────────────────┘
│(File/Subproc)
└──────┬──────┘
       │ Verified & Tests Pass
       ▼
  FINAL RESULT
```

### Safety Requirement: Iteration Bound
All production loops must enforce a hard iteration boundary (`MAX_ITERATIONS = 8`) to guarantee termination.

In [11]:
# 06.1 Bounded Autonomous Agent Loop
class AutonomousCodingLoop:
    def __init__(self, max_iterations: int = 8):
        self.max_iterations = max_iterations

    def run(self, goal: str) -> Dict[str, Any]:
        execution_trace = []
        status = "IN_PROGRESS"
        
        # Step 1: Formulate Plan
        execution_trace.append({"step": 1, "phase": "PLAN", "detail": f"Deconstruct goal: '{goal}' into repo actions."})
        
        # Step 2: Search Files
        execution_trace.append({"step": 2, "phase": "ACT", "tool": "search_files", "detail": "Identified backend/services/notification.py"})
        
        # Step 3: Run Initial Tests
        test_run = safe_run_terminal("pytest tests/test_notification.py")
        execution_trace.append({
            "step": 3,
            "phase": "OBSERVE",
            "tool": "pytest",
            "exit_code": test_run.exit_code,
            "observation": "Test suite failed with KeyError in dispatch()."
        })
        
        # Step 4: Self-Correction Patch
        execution_trace.append({"step": 4, "phase": "ACT", "tool": "write_file", "detail": "Patched payload validation in notification.py"})
        
        # Step 5: Verification Run
        execution_trace.append({"step": 5, "phase": "VERIFY", "tool": "pytest", "observation": "100% tests passing."})
        
        return {
            "goal": goal,
            "status": "COMPLETED",
            "total_steps": 5,
            "trace": execution_trace
        }

coding_loop = AutonomousCodingLoop(max_iterations=8)
loop_res = coding_loop.run("Fix NotificationService payload validation bug")

print("🔄 Autonomous Agent Loop Trace:")
for item in loop_res["trace"]:
    print(f"  [{item['step']}] {item['phase']:<8} ➔ {item.get('detail') or item.get('observation')}")

🔄 Autonomous Agent Loop Trace:
  [1] PLAN     ➔ Deconstruct goal: 'Fix NotificationService payload validation bug' into repo actions.
  [2] ACT      ➔ Identified backend/services/notification.py
  [3] OBSERVE  ➔ Test suite failed with KeyError in dispatch().
  [4] ACT      ➔ Patched payload validation in notification.py
  [5] VERIFY   ➔ 100% tests passing.


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 06
Without an explicit verification phase (`VERIFY`), an agent is simply executing unvetted file changes. Verification against a deterministic test runner is mandatory.

---
# 07 — ReAct-Style Reasoning

In production systems, we must maintain a clear distinction between:
- **Internal Chain-of-Thought**: Scratchpads where the model speculates.
- **Observable Agent State**: Safe, structured audit summaries emitted to the engineering log.

```text
Decision / Thought ──► Action (Tool) ──► Observation ──► Next Decision
```

In [12]:
# 07.1 ReAct Trace Formulation
class ReActLogItem(BaseModel):
    step: int
    decision: str
    action: str
    observation: str

def execute_react_step(step: int, decision: str, action: str, observation: str) -> ReActLogItem:
    return ReActLogItem(
        step=step,
        decision=decision,
        action=action,
        observation=observation
    )

react_trace = [
    execute_react_step(1, "Locate notification service", "search_files(query='notification')", "Found backend/services/notification.py"),
    execute_react_step(2, "Inspect current tests", "read_file(path='tests/test_notification.py')", "Test requires ValueError on missing message"),
    execute_react_step(3, "Execute test suite to reproduce", "run_terminal(command='pytest')", "Observed KeyError failure in dispatch()"),
    execute_react_step(4, "Apply input validation guard", "write_file(path='backend/services/notification.py')", "Added validation check for payload['message']")
]

print("📋 Observable ReAct Trace (Public Audit View):")
for r in react_trace:
    print(f"  Step {r.step}: Decision: {r.decision}")
    print(f"          Action:   {r.action}")
    print(f"          Observe:  {r.observation}\n")

📋 Observable ReAct Trace (Public Audit View):
  Step 1: Decision: Locate notification service
          Action:   search_files(query='notification')
          Observe:  Found backend/services/notification.py

  Step 2: Decision: Inspect current tests
          Action:   read_file(path='tests/test_notification.py')
          Observe:  Test requires ValueError on missing message

  Step 3: Decision: Execute test suite to reproduce
          Action:   run_terminal(command='pytest')
          Observe:  Observed KeyError failure in dispatch()

  Step 4: Decision: Apply input validation guard
          Action:   write_file(path='backend/services/notification.py')
          Observe:  Added validation check for payload['message']



### 💡 Engineering Takeaway — Part 07
Never stream raw reasoning tokens to client applications. Always filter internal scratchpads into structured `decision` and `observation` tuples for security and compliance auditing.

---
# 08 — Persistent Project Memory

*Context window is ephemeral working memory; repository documents are persistent corporate memory.*

Every developer team maintains guidelines (`AGENTS.md`, `architecture.md`, `decisions.md`).
Before an autonomous agent proposes any code modification, it **must ingest the project memory files** to adhere to established engineering rules.

In [13]:
# 08.1 Project Memory Loader & Policy Validator
class ProjectMemoryManager:
    def __init__(self, repo_dir: Path):
        self.repo_dir = repo_dir
        self.rules: List[str] = []
        self.tasks: List[str] = []

    def load_memory(self) -> Dict[str, Any]:
        agents_md = self.repo_dir / "AGENTS.md"
        tasks_md = self.repo_dir / "tasks.md"
        arch_md = self.repo_dir / "docs" / "architecture.md"

        memory = {
            "operating_rules": agents_md.read_text().splitlines() if agents_md.exists() else [],
            "pending_tasks": [line for line in tasks_md.read_text().splitlines() if "[ ]" in line] if tasks_md.exists() else [],
            "architecture_summary": arch_md.read_text()[:250] if arch_md.exists() else ""
        }
        return memory

memory_mgr = ProjectMemoryManager(SANDBOX_DIR)
repo_memory = memory_mgr.load_memory()

print("🧠 Persistent Project Memory Ingested:")
print(f"Active Rules Count: {len(repo_memory['operating_rules'])}")
for r in repo_memory['operating_rules'][:4]:
    print(f"  📜 {r}")

print(f"\nPending Sprint Tasks:")
for t in repo_memory['pending_tasks']:
    print(f"  🎯 {t}")

🧠 Persistent Project Memory Ingested:
Active Rules Count: 6
  📜 # DevFlow Agent Operating Rules
  📜 1. Always add type hints to all Python functions.
  📜 2. Business logic must reside in `backend/services/`, never in `backend/api/`.
  📜 3. Never swallow exceptions silently; log them with correlation IDs.

Pending Sprint Tasks:
  🎯 - [ ] Fix NotificationService payload validation bug
  🎯 - [ ] Add Email and In-App multi-channel delivery support
  🎯 - [ ] Run test suite and pass all assertions


### 💡 Engineering Takeaway — Part 08
`AGENTS.md` is to coding agents what `.eslintrc` or `pyproject.toml` is to compilers. It grounds the agent's actions in team-specific architecture, naming conventions, and compliance rules.

---
# 09 — Git-Aware Coding Agent

Git is the **human control plane** for coding agents.
By having the agent inspect `git status` and `git diff`, engineers can:
1. Verify exactly what lines the agent changed.
2. Ensure no unintended configuration or credential files were touched.
3. Use commits as atomic rollback points.

Let us initialize a local Git repository inside our sandbox.

In [14]:
# 09.1 Sandboxed Local Git Initialization & Inspection
def init_sandbox_git() -> TerminalResult:
    safe_run_terminal("git init")
    safe_run_terminal("git config user.name 'DevFlowAgent'")
    safe_run_terminal("git config user.email 'agent@novastack.com'")
    safe_run_terminal("git add .")
    return safe_run_terminal("git commit -m 'Initial commit of DevFlow sandbox'")

init_res = init_sandbox_git()
print(f"📦 Git Initialized in Sandbox: Exit code {init_res.exit_code}")

def tool_git_status() -> str:
    res = safe_run_terminal("git status --short")
    return res.stdout if res.stdout else "Clean working tree."

def tool_git_diff() -> str:
    res = safe_run_terminal("git diff")
    return res.stdout if res.stdout else "No unstaged changes."

print("Git Status:", tool_git_status())

/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


📦 Git Initialized in Sandbox: Exit code 0
Git Status: Clean working tree.


### 💡 Engineering Takeaway — Part 09
Never let an agent perform automated `git push --force` to remote branches. Git diffs provide the primary review surface before human sign-off.

---
# 10 — Self-Correcting Coding Agent

Here we implement the complete **Self-Correction Feedback Loop**:
```text
1. Execute pytest ──► Fails with KeyError
2. Parse Traceback ──► Identify bug in backend/services/notification.py
3. Synthesize Patch ──► Write defensive validation check
4. Re-run pytest ──► Verify 100% test pass rate
```

In [15]:
# 10.1 Running Initial Failing Test
initial_test_run = safe_run_terminal("pytest tests/test_notification.py")

print(f"❌ Initial Test Run Result: Exit Code {initial_test_run.exit_code}")
print("Traceback Snippet:\n" + "\n".join(initial_test_run.stdout.splitlines()[-12:]))

❌ Initial Test Run Result: Exit Code 1
Traceback Snippet:

    def dispatch(self, user_id: str, payload: Dict[str, Any]) -> bool:
        # BUG: Intentionally missing validation for empty message content!
        # When payload['message'] is missing or None, raises KeyError instead of ValueError
>       message_body = payload["message"]
                       ^^^^^^^^^^^^^^^^^^
E       KeyError: 'message'

backend/services/notification.py:11: KeyError
=========================== short test summary info ============================
FAILED tests/test_notification.py::test_missing_message_raises_value_error - KeyError: 'message'
========================= 1 failed, 1 passed in 0.03s ==========================


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


In [16]:
# 10.2 Diagnosing and Applying Code Patch
def diagnose_and_patch_bug():
    print("🛠️ Self-Correcting Coding Agent: Diagnosing traceback...")
    print("   Detected: KeyError: 'message' when payload lacks 'message' key.")
    print("   Resolution: Enforce validation: raise ValueError if message is missing or empty.")

    # High-quality, robust fix adhering to AGENTS.md
    fixed_code = """# Notification Service Implementation (Patched by DevFlow Agent)
from typing import Dict, Any, Optional

class NotificationService:
    '''
    Enterprise multi-channel notification dispatcher for DevFlow.
    Conforms to NovaStack AGENTS.md standards with strict type checking.
    '''
    def __init__(self):
        self.sent_history: list[Dict[str, Any]] = []

    def dispatch(self, user_id: str, payload: Dict[str, Any]) -> bool:
        if not payload or "message" not in payload or not payload["message"]:
            raise ValueError("Payload must contain a non-empty 'message' string.")
            
        message_body = str(payload["message"]).strip()
        channel = payload.get("channel", "in_app")
        
        record = {
            "user_id": user_id,
            "message": message_body,
            "channel": channel,
            "status": "DELIVERED"
        }
        self.sent_history.append(record)
        return True
"""
    # Write patched code
    (SANDBOX_DIR / "backend" / "services" / "notification.py").write_text(fixed_code, encoding="utf-8")
    print("✅ Patch successfully written to backend/services/notification.py")

diagnose_and_patch_bug()

🛠️ Self-Correcting Coding Agent: Diagnosing traceback...
   Detected: KeyError: 'message' when payload lacks 'message' key.
   Resolution: Enforce validation: raise ValueError if message is missing or empty.
✅ Patch successfully written to backend/services/notification.py


In [17]:
# 10.3 Verifying Test Suite After Patch
verification_run = safe_run_terminal("pytest tests/test_notification.py")

print(f"🎉 Post-Patch Verification Test: Exit Code {verification_run.exit_code}")
print("Stdout Output:\n" + "\n".join(verification_run.stdout.splitlines()[-8:]))
print("\n🔍 Git Diff of the Autonomous Patch:")
print(tool_git_diff())

🎉 Post-Patch Verification Test: Exit Code 0
Stdout Output:
platform darwin -- Python 3.13.7, pytest-9.1.1, pluggy-1.6.0
rootdir: /private/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/devflow_repo_c6s66v77
configfile: pyproject.toml
collected 2 items

tests/test_notification.py ..                                            [100%]

============================== 2 passed in 0.00s ===============================

🔍 Git Diff of the Autonomous Patch:
diff --git a/backend/services/__pycache__/notification.cpython-313.pyc b/backend/services/__pycache__/notification.cpython-313.pyc
index 63ea323..1fed638 100644
Binary files a/backend/services/__pycache__/notification.cpython-313.pyc and b/backend/services/__pycache__/notification.cpython-313.pyc differ
diff --git a/backend/services/notification.py b/backend/services/notification.py
index f7cea8f..b90fbcb 100644
--- a/backend/services/notification.py
+++ b/backend/services/notification.py
@@ -1,19 +1,25 @@
-# Notification Service Implementat

/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 10
*Self-correction is the defining capability of coding agents.* A model that can observe its own runtime errors and iteratively fix them is orders of magnitude more reliable than any one-shot prompt generator.

---
# 11 — AI Code Review Agent

An autonomous engineer should not approve its own code without an independent review.
We implement an **AI Reviewer Agent** that inspects:
1. `git_diff()`
2. Test pass confirmation
3. Compliance with `AGENTS.md` rules
4. Security vulnerabilities (OWASP) and maintainability

The reviewer returns a structured, typed audit report.

In [18]:
# 11.1 Structured Code Review Schema and Audit Runner
class CodeReviewReport(BaseModel):
    approved: bool
    severity: str = Field(description="'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'")
    issues: List[str] = Field(default_factory=list)
    suggestions: List[str] = Field(default_factory=list)
    compliance_score: float = Field(description="Score between 0.0 and 1.0")

def run_ai_code_review(diff: str, test_passed: bool) -> CodeReviewReport:
    issues = []
    suggestions = []
    
    # Static and semantic review heuristics
    if not test_passed:
        issues.append("Blocking: Test suite did not pass.")
    if "eval(" in diff or "exec(" in diff:
        issues.append("Security Alert: Use of dynamic execution functions is forbidden.")
    if "ValueError" in diff and "type hints" in diff:
        suggestions.append("Good adherence to defensive validation and type annotations.")
        
    is_approved = (len(issues) == 0) and test_passed
    return CodeReviewReport(
        approved=is_approved,
        severity="LOW" if is_approved else "HIGH",
        issues=issues,
        suggestions=suggestions,
        compliance_score=0.96 if is_approved else 0.40
    )

active_diff = tool_git_diff()
review_result = run_ai_code_review(active_diff, test_passed=(verification_run.exit_code == 0))

print("🧐 AI Code Review Audit Report:")
print(json.dumps(review_result.model_dump(), indent=2))
print(f"\nReview Status: {'✅ APPROVED' if review_result.approved else '❌ REJECTED'}")

🧐 AI Code Review Audit Report:
{
  "approved": true,
  "severity": "LOW",
  "issues": [],
  "suggestions": [],
  "compliance_score": 0.96
}

Review Status: ✅ APPROVED


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 11
Decouple the Developer Agent from the Reviewer Agent. A dedicated reviewer prompt with a strict audit mandate catches edge cases that the implementation agent overlooks.

---
# 12 — Multi-Agent Engineering Team

Real software development involves multiple roles collaborating:
- **Supervisor Agent**: Breaks large tasks into atomic sub-tasks and delegates.
- **Research Agent**: Ingests repository documentation, dependencies, and PRDs.
- **Developer Agent**: Writes code and implements features.
- **Reviewer Agent**: Audits security, style, and regression risks.
- **QA Agent**: Executes test suites and validates edge cases.

```text
                 SUPERVISOR
                     │
        ┌────────────┼────────────┐
        ▼            ▼            ▼
    RESEARCH      DEVELOPER       QA
        │            │            │
        └────────────┼────────────┘
                     ▼
                  REVIEWER
                     │
                     ▼
                 FINAL RESULT
```

In [19]:
# 12.1 Multi-Agent Engineering Workforce Coordination
@dataclass
class TeamAgent:
    name: str
    role: str
    system_prompt: str

ENGINEERING_TEAM: Dict[str, TeamAgent] = {
    "Supervisor": TeamAgent(
        name="Alex (Supervisor)",
        role="Engineering Manager",
        system_prompt="Decompose engineering epics into tasks, assign to specialists, and track delivery."
    ),
    "Researcher": TeamAgent(
        name="Elena (Researcher)",
        role="Domain & Repo Architect",
        system_prompt="Analyze codebases, locate relevant modules, and extract interface contracts."
    ),
    "Developer": TeamAgent(
        name="Devin (Developer)",
        role="Full-Stack Engineer",
        system_prompt="Write production-grade code, implement interfaces, and generate unit tests."
    ),
    "QA": TeamAgent(
        name="Quinn (QA)",
        role="Test & Quality Lead",
        system_prompt="Run pytest, stress-test boundary conditions, and verify regression safety."
    ),
    "Reviewer": TeamAgent(
        name="Riley (Reviewer)",
        role="Principal Security Reviewer",
        system_prompt="Audit git diffs, ensure compliance with AGENTS.md, and approve or reject PRs."
    )
}

print(f"👥 Multi-Agent Team Assembled ({len(ENGINEERING_TEAM)} Specialists):")
for k, agent in ENGINEERING_TEAM.items():
    print(f"  • {agent.name:<20} | Role: {agent.role}")

👥 Multi-Agent Team Assembled (5 Specialists):
  • Alex (Supervisor)    | Role: Engineering Manager
  • Elena (Researcher)   | Role: Domain & Repo Architect
  • Devin (Developer)    | Role: Full-Stack Engineer
  • Quinn (QA)           | Role: Test & Quality Lead
  • Riley (Reviewer)     | Role: Principal Security Reviewer


### 💡 Engineering Takeaway — Part 12
Specialization keeps prompt context focused. If the developer agent does not need to worry about high-level project schedules, it can dedicate its entire context window to AST and code implementation details.

---
# 13 — Cross-Repository Engineering

In modern microservice architectures, enterprise features span multiple repositories:
- `devflow-frontend` (Next.js / TypeScript UI)
- `devflow-backend` (FastAPI Python services)
- `devflow-mobile` (React Native client)
- `devflow-infrastructure` (Terraform / Kubernetes manifests)

Let us simulate a cross-repository task:
> **“Add OAuth 2.0 authentication support across frontend, backend, mobile, and infrastructure.”**

In [20]:
# 13.1 Cross-Repository Dependency Graph and Orchestrator
CROSS_REPOS = {
    "devflow-backend": {"lang": "Python", "role": "Auth endpoints (/oauth/token, /oauth/verify)", "depends_on": []},
    "devflow-infrastructure": {"lang": "Terraform", "role": "Deploy OAuth proxy & secrets", "depends_on": []},
    "devflow-frontend": {"lang": "TypeScript", "role": "Login page & token storage", "depends_on": ["devflow-backend"]},
    "devflow-mobile": {"lang": "React Native", "role": "OAuth deep-linking handler", "depends_on": ["devflow-backend"]}
}

class CrossRepoOrchestrator:
    def __init__(self, repo_catalog: Dict[str, Any]):
        self.catalog = repo_catalog

    def plan_cross_repo_rollout(self, feature_name: str) -> List[Dict[str, Any]]:
        plan = []
        # Stage 1: Foundational Backend & Infrastructure
        plan.append({
            "stage": 1,
            "phase": "Foundational Services",
            "actions": [
                {"repo": "devflow-backend", "task": "Implement OAuth2 RFC 6749 endpoints"},
                {"repo": "devflow-infrastructure", "task": "Provision OAuth client IDs in Vault"}
            ]
        })
        # Stage 2: Dependent Client Apps
        plan.append({
            "stage": 2,
            "phase": "Client Integrations",
            "actions": [
                {"repo": "devflow-frontend", "task": "Integrate PKCE authorization code flow"},
                {"repo": "devflow-mobile", "task": "Register custom URI scheme for OAuth redirect"}
            ]
        })
        return plan

orchestrator = CrossRepoOrchestrator(CROSS_REPOS)
cross_plan = orchestrator.plan_cross_repo_rollout("OAuth 2.0 Support")

print("🌐 Cross-Repository Rollout Plan:")
for stage in cross_plan:
    print(f"  Stage {stage['stage']} [{stage['phase']}]:")
    for act in stage["actions"]:
        print(f"    ➜ {act['repo']:<25}: {act['task']}")

🌐 Cross-Repository Rollout Plan:
  Stage 1 [Foundational Services]:
    ➜ devflow-backend          : Implement OAuth2 RFC 6749 endpoints
    ➜ devflow-infrastructure   : Provision OAuth client IDs in Vault
  Stage 2 [Client Integrations]:
    ➜ devflow-frontend         : Integrate PKCE authorization code flow
    ➜ devflow-mobile           : Register custom URI scheme for OAuth redirect


### 💡 Engineering Takeaway — Part 13
Cross-repository engineering requires topological dependency ordering. The agent supervisor must sequence foundational backend and infrastructure changes before generating dependent frontend and mobile client code.

---
# 14 — Automated Testing & Verification

*Autonomous execution without verification is automated liability.*

A robust agent pipeline executes layered verification gates:
1. **Unit Tests**: Function-level contracts.
2. **Integration Tests**: Service interactions.
3. **Static Checks**: Type checking (`mypy`) and syntax validation.
4. **Agent Semantic Verification**: Verifying that requirements were fully addressed.

In [21]:
# 14.1 Layered Verification Suite Runner
class VerificationSuite:
    @staticmethod
    def run_unit_tests() -> Dict[str, Any]:
        res = safe_run_terminal("pytest tests/test_notification.py")
        return {"suite": "unit_tests", "passed": res.exit_code == 0, "exit_code": res.exit_code}

    @staticmethod
    def run_static_syntax_check() -> Dict[str, Any]:
        # Compile check across python files
        py_files = list(SANDBOX_DIR.rglob("*.py"))
        syntax_errors = []
        for p in py_files:
            try:
                compile(p.read_text(encoding="utf-8"), str(p), "exec")
            except SyntaxError as se:
                syntax_errors.append(f"{p.name}: {se}")
        return {"suite": "syntax_compilation", "passed": len(syntax_errors) == 0, "errors": syntax_errors}

    @staticmethod
    def execute_all() -> pd.DataFrame:
        results = [
            VerificationSuite.run_static_syntax_check(),
            VerificationSuite.run_unit_tests()
        ]
        return pd.DataFrame(results)

df_verify = VerificationSuite.execute_all()
print("🛡️ Automated Verification Suite Results:")
print(df_verify.to_string(index=False))

🛡️ Automated Verification Suite Results:
             suite  passed errors  exit_code
syntax_compilation    True     []        NaN
        unit_tests    True    NaN        0.0


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 14
Layered testing prevents syntax errors from wasting time in unit tests, and prevents unit test passes from hiding architectural integration regressions.

---
# 15 — Agentic CI/CD Pipeline

When an agent finishes work, it commits changes and submits a pull request triggering an automated **Agentic CI/CD Pipeline**:

```text
Git Change ──► CI Trigger ──► Lint ──► Tests ──► Security Scan ──► AI Audit ──► Deploy Sim
```

In [22]:
# 15.1 Simulated Agentic CI/CD Pipeline
class AgenticCIData(BaseModel):
    pipeline_id: str
    git_commit: str
    stages: List[Dict[str, str]]
    final_status: str
    deployment_eligible: bool

def run_cicd_pipeline(commit_sha: str = "c9a184e") -> AgenticCIData:
    stages = []
    
    # Stage 1: Lint & Code Style
    stages.append({"stage": "Linting", "status": "PASSED", "detail": "PEP8 & Black formatted"})
    
    # Stage 2: Automated Tests
    unit_res = VerificationSuite.run_unit_tests()
    stages.append({"stage": "Pytest Suite", "status": "PASSED" if unit_res["passed"] else "FAILED", "detail": "2/2 tests green"})
    
    # Stage 3: Security & Secret Scan
    stages.append({"stage": "Secret Scanner", "status": "PASSED", "detail": "Zero credentials detected in diff"})
    
    # Stage 4: AI Review Audit
    stages.append({"stage": "AI Review Gate", "status": "PASSED", "detail": "Compliance score 0.96 / 1.0"})
    
    all_passed = all(s["status"] == "PASSED" for s in stages)
    
    return AgenticCIData(
        pipeline_id=f"CI-{uuid.uuid4().hex[:6]}",
        git_commit=commit_sha,
        stages=stages,
        final_status="SUCCESS" if all_passed else "FAILED",
        deployment_eligible=all_passed
    )

cicd_run = run_cicd_pipeline()
print(f"🚀 CI/CD Pipeline {cicd_run.pipeline_id} for Commit #{cicd_run.git_commit}:")
for s in cicd_run.stages:
    print(f"   [{s['status']}] {s['stage']:<20} ➔ {s['detail']}")

print(f"\nPipeline Result: {cicd_run.final_status} (Deployment Eligible: {cicd_run.deployment_eligible})")

🚀 CI/CD Pipeline CI-865b12 for Commit #c9a184e:
   [PASSED] Linting              ➔ PEP8 & Black formatted
   [PASSED] Pytest Suite         ➔ 2/2 tests green
   [PASSED] Secret Scanner       ➔ Zero credentials detected in diff
   [PASSED] AI Review Gate       ➔ Compliance score 0.96 / 1.0

Pipeline Result: SUCCESS (Deployment Eligible: True)


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 15
CI/CD is not just for human engineers; it is the ultimate automated supervisor for AI coding agents. If the CI/CD pipeline fails, the failure logs are fed back into the agent loop for automated resolution.

---
# 16 — Human-in-the-Loop (HITL)

Autonomy should be governed by risk:
- **READ** operations: Automatic.
- **WRITE** operations: Automatic inside sandboxes.
- **EXECUTE** operations: Restricted by allowlists.
- **DEPLOY** operations: **Mandatory Human Approval.**

Let us build a strict **HITL Policy Engine**.

In [23]:
# 16.1 Policy Engine & Human Approval Interceptor
class PermissionLevel:
    READ = "READ"
    WRITE = "WRITE"
    EXECUTE = "EXECUTE"
    DEPLOY = "DEPLOY"

class HITLPolicyEngine:
    def __init__(self, auto_approve_deploy: bool = False):
        self.auto_approve_deploy = auto_approve_deploy

    def check_permission(self, action_type: str, target: str) -> Dict[str, Any]:
        if action_type in [PermissionLevel.READ, PermissionLevel.WRITE]:
            return {"authorized": True, "action": action_type, "gate": "AUTOMATIC"}
        elif action_type == PermissionLevel.EXECUTE:
            return {"authorized": True, "action": action_type, "gate": "SANDBOX_RESTRICTED"}
        elif action_type == PermissionLevel.DEPLOY:
            # Dangerous production mutation: HALT and ask human
            print(f"\n🛑 [HITL INTERCEPTOR] Agent requested action: '{action_type}' on '{target}'")
            print("   Action is classified as IRREVERSIBLE / HIGH IMPACT.")
            decision = "APPROVED" if self.auto_approve_deploy else "REJECTED"
            reason = "Human SRE verified staging test receipt." if decision == "APPROVED" else "Deployment rejected: Human sign-off required on production release."
            return {
                "authorized": decision == "APPROVED",
                "action": action_type,
                "gate": "HUMAN_APPROVAL_REQUIRED",
                "decision": decision,
                "reason": reason
            }
        return {"authorized": False, "reason": "Unknown permission tier"}

hitl = HITLPolicyEngine(auto_approve_deploy=False)

# Test automatic vs intercepted permissions
p_read = hitl.check_permission(PermissionLevel.READ, "backend/services/notification.py")
print(f"Read Permission: Authorized={p_read['authorized']} (Gate: {p_read['gate']})")

p_deploy = hitl.check_permission(PermissionLevel.DEPLOY, "production-k8s-cluster")
print(f"Deploy Permission: Authorized={p_deploy['authorized']} (Decision: {p_deploy['decision']})")

Read Permission: Authorized=True (Gate: AUTOMATIC)

🛑 [HITL INTERCEPTOR] Agent requested action: 'DEPLOY' on 'production-k8s-cluster'
   Action is classified as IRREVERSIBLE / HIGH IMPACT.
Deploy Permission: Authorized=False (Decision: REJECTED)


### 💡 Engineering Takeaway — Part 16
*Human-in-the-loop is an engineering guardrail, not a bottleneck.* By automating low-risk reads and sandbox writes while requiring explicit sign-off on deployments, engineering teams achieve high velocity with zero accidental outages.

---
# 17 — Agent Observability

To debug, audit, and optimize coding agents in production, every action must be recorded as a structured **Agent Event**:
```text
timestamp | agent | task_id | action | tool | arguments_summary | latency | tokens | status | error
```

In [24]:
# 17.1 Structured Agent Event Model and Observability Logger
class AgentEvent(BaseModel):
    event_id: str
    timestamp: str
    agent: str
    task_id: str
    action: str
    tool: str
    arguments_summary: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    status: str
    error: Optional[str] = None

class ObservabilityCollector:
    def __init__(self):
        self.events: List[AgentEvent] = []

    def log(self, agent: str, task_id: str, action: str, tool: str,
            args_summary: str, latency_ms: float, in_tokens: int, out_tokens: int, status: str = "SUCCESS", error: Optional[str] = None):
        event = AgentEvent(
            event_id=f"EVT-{uuid.uuid4().hex[:6]}",
            timestamp=datetime.utcnow().isoformat(),
            agent=agent,
            task_id=task_id,
            action=action,
            tool=tool,
            arguments_summary=args_summary,
            latency_ms=latency_ms,
            input_tokens=in_tokens,
            output_tokens=out_tokens,
            status=status,
            error=error
        )
        self.events.append(event)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([e.model_dump() for e in self.events])

# Populate realistic telemetry audit trail
obs = ObservabilityCollector()
tid = "TASK-NOTIFICATION-FIX"

obs.log("Supervisor", tid, "PLAN", "none", "Decompose sprint goal", 110.5, 450, 120)
obs.log("Researcher", tid, "SEARCH", "search_files", "query='notification'", 35.2, 0, 0)
obs.log("Developer", tid, "TEST_RUN", "pytest", "tests/test_notification.py", 280.0, 0, 0, status="FAILURE", error="KeyError: 'message'")
obs.log("Developer", tid, "PATCH", "write_file", "backend/services/notification.py", 14.1, 0, 0)
obs.log("QA", tid, "VERIFY", "pytest", "tests/test_notification.py", 260.4, 0, 0)
obs.log("Reviewer", tid, "AUDIT", "git_diff", "Inspect patch diff", 145.0, 520, 180)

df_obs = obs.to_dataframe()
print(f"📊 Observability Audit Trail for Task: {tid}")
print(df_obs[["timestamp", "agent", "action", "tool", "latency_ms", "status"]].to_string(index=False))

📊 Observability Audit Trail for Task: TASK-NOTIFICATION-FIX
                 timestamp      agent   action         tool  latency_ms  status
2026-09-05T18:37:31.397200 Supervisor     PLAN         none       110.5 SUCCESS
2026-09-05T18:37:31.397249 Researcher   SEARCH search_files        35.2 SUCCESS
2026-09-05T18:37:31.397289  Developer TEST_RUN       pytest       280.0 FAILURE
2026-09-05T18:37:31.397318  Developer    PATCH   write_file        14.1 SUCCESS
2026-09-05T18:37:31.397346         QA   VERIFY       pytest       260.4 SUCCESS
2026-09-05T18:37:31.397371   Reviewer    AUDIT     git_diff       145.0 SUCCESS


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/2251568762.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat(),


### 💡 Engineering Takeaway — Part 17
Without structured event logging, debugging multi-agent systems is impossible. Event telemetry allows you to reconstruct the exact timeline and diagnose why an agent took a wrong turn.

---
# 18 — Agent Performance Metrics

Engineering teams manage what they measure.
For autonomous coding agents, key quantitative performance indicators include:
- **Task Success Rate**: `successful_tasks / total_tasks`
- **Tool Success Rate**: `successful_tool_calls / total_tool_calls`
- **Retry Rate**: `retries / total_tasks`
- **Mean Task Latency**: `total_task_time / completed_tasks`
- **Cost Efficiency**: Total token expenditure per successful feature PR.

In [25]:
# 18.1 Agent Performance Dashboard Calculator
class AgentMetricsEngine:
    @staticmethod
    def compute_kpis(events_df: pd.DataFrame) -> Dict[str, Any]:
        total_actions = len(events_df)
        successful_actions = len(events_df[events_df["status"] == "SUCCESS"])
        tool_success_rate = round((successful_actions / total_actions) * 100, 1) if total_actions > 0 else 0.0
        
        total_latency_sec = round(events_df["latency_ms"].sum() / 1000, 2)
        
        # Token calculation: $2.50/1M in, $10.00/1M out
        total_tokens = events_df["input_tokens"].sum() + events_df["output_tokens"].sum()
        total_cost_usd = round((events_df["input_tokens"].sum() * 0.0000025) + (events_df["output_tokens"].sum() * 0.00001), 5)
        
        return {
            "Total Agent Actions": total_actions,
            "Tool Success Rate": f"{tool_success_rate}%",
            "Cumulative Latency (s)": total_latency_sec,
            "Total Tokens Consumed": total_tokens,
            "Estimated Cost (USD)": f"${total_cost_usd:.5f}",
            "Autonomous Task Success Rate": "100.0%"
        }

metrics_summary = AgentMetricsEngine.compute_kpis(df_obs)
df_kpis = pd.DataFrame(list(metrics_summary.items()), columns=["Metric", "Value"])

print("📈 Autonomous Engineering Team KPI Dashboard:")
print(df_kpis.to_string(index=False))

📈 Autonomous Engineering Team KPI Dashboard:
                      Metric    Value
         Total Agent Actions        6
           Tool Success Rate    83.3%
      Cumulative Latency (s)     0.85
       Total Tokens Consumed     1270
        Estimated Cost (USD) $0.00542
Autonomous Task Success Rate   100.0%


### 💡 Engineering Takeaway — Part 18
Track cost per merged pull request. If an autonomous agent spends $0.04 in tokens to diagnose and patch an issue that takes an engineer 30 minutes of debugging, the ROI is over 100x.

---
# 19 — Failure Engineering

Production coding agents will break unless they are explicitly hardened against edge cases.

### The 8 Classical Failure Modes in Coding Agents:
1. **Missing Files**: Attempting to read non-existent paths.
2. **Invalid Tool Arguments**: Model passes strings instead of integers or dictionaries.
3. **Test Failures**: Syntax errors, regressions, unhandled exceptions.
4. **Subprocess Timeouts**: Infinite loops in tests or build scripts.
5. **Permission Denial**: Path traversal attempts or unauthorized binaries.
6. **Malformed Model Output**: Unparseable JSON or empty tool calls.
7. **Infinite Agent Loops**: Repeatedly calling the same tool with identical inputs.
8. **Stale Project Memory**: Modifying files based on outdated documentation.

Let us test our system's automated recovery across these scenarios.

In [26]:
# 19.1 Failure Simulation and Defense Verification
print("--- Scenario 1: Handling Non-Existent File Read ---")
missing_read = tool_read_file("backend/services/non_existent.py")
print("Agent Observation:", missing_read)

print("\n--- Scenario 2: Defense Against Path Traversal Attack ---")
try:
    tool_write_file(WriteFileArgs(path="../../../etc/passwd", content="malicious"))
except Exception as e:
    print("Security Interception:", e)

print("\n--- Scenario 3: Subprocess Timeout Protection ---")
# Simulating command that takes too long
timeout_test = safe_run_terminal("python -c 'import time; time.sleep(20)'", timeout_seconds=1)
print("Timeout Interception:", timeout_test.stderr)

print("\n--- Scenario 4: Infinite Loop Breaker Defense ---")
def check_loop_trap(history: List[str], current: str) -> bool:
    repeats = history.count(current)
    return repeats >= 2

call_history = ["search:notification", "search:notification"]
trapped = check_loop_trap(call_history, "search:notification")
print(f"Infinite loop trap triggered? {trapped} ➔ Force abort & alert supervisor!")

--- Scenario 1: Handling Non-Existent File Read ---
Agent Observation: Error: File 'backend/services/non_existent.py' does not exist.

--- Scenario 2: Defense Against Path Traversal Attack ---
Security Interception: Security Alert: Attempted path traversal to '../../../etc/passwd'.

--- Scenario 3: Subprocess Timeout Protection ---


Timeout Interception: Timeout: Command exceeded 1s limit.

--- Scenario 4: Infinite Loop Breaker Defense ---
Infinite loop trap triggered? True ➔ Force abort & alert supervisor!


/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


### 💡 Engineering Takeaway — Part 19
*Production agent engineering is largely failure engineering.* Systems that anticipate missing files, enforce subprocess timeouts, and detect loop traps remain reliable under adversarial or unexpected inputs.

---
# 20 — Cost & Performance Engineering

Agent latency compounds with every step:
```text
Total Latency = LLM Inference Latency + File I/O + Subprocess Test Execution + Review Delay
```

### The Engineering Trade-Off Space:
```text
More Autonomy  ──►  More Tool Iterations  ──►  Higher Latency & Token Cost
More Strictness ──►  More Verification Gates ──►  Higher Reliability & Safety
```

In [27]:
# 20.1 Architecture Trade-Off Profiler
latency_profiles = [
    {
        "Paradigm": "Single-Pass Copilot",
        "Iterations": 1,
        "Avg Latency (s)": 0.8,
        "Avg Tokens": 350,
        "Human Effort": "High (Review & debug)",
        "Verification": "Zero"
    },
    {
        "Paradigm": "Single ReAct Agent",
        "Iterations": 4,
        "Avg Latency (s)": 3.2,
        "Avg Tokens": 1800,
        "Human Effort": "Moderate (Check diff)",
        "Verification": "Automated Pytest"
    },
    {
        "Paradigm": "Multi-Agent Team (DevFlow)",
        "Iterations": 6,
        "Avg Latency (s)": 5.4,
        "Avg Tokens": 4200,
        "Human Effort": "Low (Final sign-off only)",
        "Verification": "Pytest + AI Review + CI/CD"
    }
]

df_lat = pd.DataFrame(latency_profiles)
print("⚡ Latency, Cost & Autonomy Trade-Off Profile:")
print(df_lat.to_string(index=False))

⚡ Latency, Cost & Autonomy Trade-Off Profile:
                  Paradigm  Iterations  Avg Latency (s)  Avg Tokens              Human Effort               Verification
       Single-Pass Copilot           1              0.8         350     High (Review & debug)                       Zero
        Single ReAct Agent           4              3.2        1800     Moderate (Check diff)           Automated Pytest
Multi-Agent Team (DevFlow)           6              5.4        4200 Low (Final sign-off only) Pytest + AI Review + CI/CD


### 💡 Engineering Takeaway — Part 20
Select agent complexity based on task risk. Use single-pass autocompletion for quick inline edits, single ReAct agents for localized refactoring, and multi-agent teams for cross-repository features and mission-critical bugfixes.

---
# 21 — Final Capstone: Multi-Channel Notification Service

### The Challenge:
> **“Implement a complete multi-channel notification service for DevFlow supporting both Email and In-App notifications, conforming to `AGENTS.md`, passing all tests, passing AI code review, and requesting human sign-off before simulated production deployment.”**

Let us execute the complete AI Software Engineering Team end-to-end!

In [28]:
# 21.1 End-to-End Capstone Execution
def run_capstone_sprint() -> Dict[str, Any]:
    print("🚀 [SPRINT START] Supervisor Alex initializing epic: 'Multi-Channel Notification Service'")
    capstone_trace = []
    
    # Phase 1: Researcher inspects requirements and memory
    memory = memory_mgr.load_memory()
    capstone_trace.append("Phase 1: Researcher Elena read AGENTS.md and architecture guidelines.")
    
    # Phase 2: Developer implements multi-channel support
    capstone_code = """# DevFlow Multi-Channel Notification Service
# Implemented autonomously by NovaStack AI Engineering Team
from typing import Dict, Any, List, Optional
import logging

logger = logging.getLogger("NotificationService")

class NotificationService:
    '''
    Enterprise multi-channel notification dispatcher supporting Email and In-App delivery.
    Complies with NovaStack AGENTS.md: strict type checking, robust error handling.
    '''
    SUPPORTED_CHANNELS = {"in_app", "email"}

    def __init__(self):
        self.sent_history: List[Dict[str, Any]] = []

    def dispatch(self, user_id: str, payload: Dict[str, Any]) -> bool:
        if not payload or "message" not in payload or not payload["message"]:
            raise ValueError("Payload must contain a non-empty 'message' string.")
            
        channel = payload.get("channel", "in_app").lower()
        if channel not in self.SUPPORTED_CHANNELS:
            raise ValueError(f"Unsupported channel '{channel}'. Allowed: {self.SUPPORTED_CHANNELS}")
            
        record = {
            "user_id": user_id,
            "message": str(payload["message"]).strip(),
            "channel": channel,
            "recipient_email": payload.get("email") if channel == "email" else None,
            "status": "DELIVERED"
        }
        self.sent_history.append(record)
        return True
"""
    (SANDBOX_DIR / "backend" / "services" / "notification.py").write_text(capstone_code, encoding="utf-8")
    capstone_trace.append("Phase 2: Developer Devin implemented multi-channel dispatch (Email & In-App).")
    
    # Phase 3: QA updates and executes test suite
    test_update = """import pytest
from backend.services.notification import NotificationService

def test_in_app_dispatch():
    svc = NotificationService()
    res = svc.dispatch("usr_01", {"message": "Build completed", "channel": "in_app"})
    assert res is True
    assert svc.sent_history[0]["channel"] == "in_app"

def test_email_dispatch():
    svc = NotificationService()
    res = svc.dispatch("usr_02", {"message": "Incident P0", "channel": "email", "email": "sre@novastack.com"})
    assert res is True
    assert svc.sent_history[0]["recipient_email"] == "sre@novastack.com"

def test_invalid_channel_raises_error():
    svc = NotificationService()
    with pytest.raises(ValueError):
        svc.dispatch("usr_03", {"message": "Hello", "channel": "sms"})
"""
    (SANDBOX_DIR / "tests" / "test_notification.py").write_text(test_update, encoding="utf-8")
    test_exec = safe_run_terminal("pytest tests/test_notification.py")
    capstone_trace.append(f"Phase 3: QA Quinn executed pytest suite. Result: {'PASS' if test_exec.exit_code == 0 else 'FAIL'} (Exit code {test_exec.exit_code})")
    
    # Phase 4: Reviewer inspects Git Diff
    diff = tool_git_diff()
    review = run_ai_code_review(diff, test_passed=(test_exec.exit_code == 0))
    capstone_trace.append(f"Phase 4: Reviewer Riley audited diff. Approved: {review.approved} (Compliance: {review.compliance_score * 100}%)")
    
    # Phase 5: CI/CD Pipeline
    ci = run_cicd_pipeline()
    capstone_trace.append(f"Phase 5: Agentic CI/CD Pipeline {ci.pipeline_id} finished: {ci.final_status}")
    
    # Phase 6: Human Approval Gate
    hitl_decision = hitl.check_permission(PermissionLevel.DEPLOY, "production-notification-service")
    capstone_trace.append(f"Phase 6: Human-in-the-Loop Gate: {hitl_decision['gate']} ➔ {hitl_decision['decision']}")
    
    return {
        "status": "SPRINT_DELIVERED",
        "trace": capstone_trace,
        "review": review,
        "ci_cd": ci
    }

capstone_run = run_capstone_sprint()
print("\n🏁 Autonomous Software Engineering Team Sprint Execution:")
for step in capstone_run["trace"]:
    print(f"  • {step}")

/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/ipykernel_6311/3720130452.py:50: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.utcnow().isoformat()


🚀 [SPRINT START] Supervisor Alex initializing epic: 'Multi-Channel Notification Service'



🛑 [HITL INTERCEPTOR] Agent requested action: 'DEPLOY' on 'production-notification-service'
   Action is classified as IRREVERSIBLE / HIGH IMPACT.

🏁 Autonomous Software Engineering Team Sprint Execution:
  • Phase 1: Researcher Elena read AGENTS.md and architecture guidelines.
  • Phase 2: Developer Devin implemented multi-channel dispatch (Email & In-App).
  • Phase 3: QA Quinn executed pytest suite. Result: PASS (Exit code 0)
  • Phase 4: Reviewer Riley audited diff. Approved: True (Compliance: 96.0%)
  • Phase 5: Agentic CI/CD Pipeline CI-53d8e5 finished: SUCCESS
  • Phase 6: Human-in-the-Loop Gate: HUMAN_APPROVAL_REQUIRED ➔ REJECTED


---
# 22 — Final Architecture Blueprint

```text
                         HUMAN ENGINEER
                               │
                               ▼
                        ┌─────────────┐
                        │ SUPERVISOR  │
                        │    AGENT    │
                        └──────┬──────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
       ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
       │  RESEARCH   │  │  DEVELOPER  │  │     QA      │
       │    AGENT    │  │    AGENT    │  │    AGENT    │
       └──────┬──────┘  └──────┬──────┘  └──────┬──────┘
              │                │                │
              └────────────────┼────────────────┘
                               ▼
                        ┌─────────────┐
                        │  REVIEWER   │
                        │    AGENT    │
                        └──────┬──────┘
                               │
                               ▼
                        ┌─────────────┐
                        │ TOOL LAYER  │
                        └──────┬──────┘
                               │
       ┌───────────┬───────────┼───────────┬───────────┐
       ▼           ▼           ▼           ▼           ▼
   ┌───────┐  ┌──────────┐ ┌───────┐  ┌─────────┐ ┌─────────┐
   │ Files │  │ Terminal │ │  Git  │  │ Pytest  │ │  CI/CD  │
   └───┬───┘  └────┬─────┘ └───┬───┘  └────┬────┘ └────┬────┘
       │           │           │           │           │
       └───────────┴───────────┼───────────┴───────────┘
                               ▼
                     ┌───────────────────┐
                     │   OBSERVABILITY   │
                     │  (Events/Traces)  │
                     └─────────┬─────────┘
                               │
                               ▼
                     ┌───────────────────┐
                     │   POLICY ENGINE   │
                     │  (HITL Clearance) │
                     └─────────┬─────────┘
                               │
                               ▼
                     ┌───────────────────┐
                     │  HUMAN APPROVAL   │
                     │ (Production Gate) │
                     └─────────┬─────────┘
                               │
                               ▼
                           DEPLOYMENT
```

---
# 23 — Final Engineering Takeaways

### The Evolution of AI-Assisted Development:
```text
Code Completion
      ↓
AI Coding Assistant (Prompt ➔ Code)
      ↓
Repository-Aware Coding Agent
      ↓
Terminal / CLI Coding Agent
      ↓
Self-Correcting Coding Agent
      ↓
AI Code Review Agent
      ↓
Multi-Agent Engineering Team
      ↓
Agentic CI/CD & Human Governance
```

### The Definitive Mental Model:
```text
Coding Agent = Model + Context + Tools + Execution + Verification + State + Memory + Observability + Human Governance
```

### The Core Architectural Principle:
> **The future of AI-assisted development is not simply better code generation. It is the engineering of reliable feedback loops around models.**

---
### Clean Up Sandbox Repository

In [29]:
# Clean up temporary sandbox directory
if SANDBOX_DIR.exists():
    shutil.rmtree(SANDBOX_DIR)
    print(f"🧹 Temporary Sandbox Directory Cleaned Up: {SANDBOX_DIR}")

🧹 Temporary Sandbox Directory Cleaned Up: /private/var/folders/9s/yp0mt3991pj5_gkjjlfyglrr0000gn/T/devflow_repo_c6s66v77
